# Phase 2 — Hybrid Deep Learning + Classical Machine Learning
## Member 2: Hybrid Innovation & Systematic Architecture

This notebook demonstrates the end-to-end implementation and empirical validation of **Hybrid Deep Learning + Machine Learning** for indoor Wi-Fi positioning on the UJIIndoorLoc benchmark.

### Core Research Question
> *Does combining the orthogonal partitioning strength of tree ensembles with the continuous manifold parameterization of deep multitask networks improve building, floor, and coordinate localization accuracy?*

### The Two Hybrid Innovations
1. **Hybrid Approach A (Class Probability Stacking):** Using 5-fold Out-Of-Fold (OOF) cross-validation, classical Random Forest classifiers generate calibrated class posterior probability vectors $P(\text{Building})$ and $P(\text{Floor})$. These structural priors are concatenated with normalized RSSI features to form a 467-dimensional input tensor for the shared deep neural trunk.
2. **Hybrid Approach B (Spatial Residual Coordinate Correction):** A regularized Histogram Gradient Boosted Decision Tree (H-GBDT) regressor models the non-linear spatial residuals ($e_x = x - \hat{x}_{\text{DL}}$, $e_y = y - \hat{y}_{\text{DL}}$) to eliminate local multipath and antenna calibration biases.
3. **Full Combined Hybrid:** The combination of both techniques achieves state-of-the-art positioning performance on the official held-out test set ($N=1{,}111$ fingerprints).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.data_preprocessing import prepare_data
from src.models.hybrid_model import HybridPositioningSystem
from src.models.multitask_net import ALL_TASKS, MultitaskNet
from src.stacking_features import TreeProbabilityExtractor, concatenate_hybrid_features
from src.residual_correction import ResidualCoordinateCorrector
from src.train_multitask import predict, compute_metrics

REPO_ROOT = Path(".") if Path("results").exists() else Path("..")
FIG_DIR = REPO_ROOT / "results" / "figures"
METRICS_DIR = REPO_ROOT / "results" / "metrics"
MODEL_DIR = REPO_ROOT / "models"
print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Inference Device:", device)

/Users/harshita/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/8q/4jf0x5gn2cb5kq8vd9v4ch800000gn/T/matplotlib-0xo3goxt because there was an issue with the default path (/Users/harshita/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.
PyTorch version: 2.14.1
Inference Device: cpu


## 1. Load Data Splits and Preprocessing Pipeline
All models are evaluated on the exact same train/validation/test split:
- **Train (85%):** 16,432 fingerprints (GroupShuffleSplit by position ID to prevent spatial leakage)
- **Validation (15%):** 2,795 fingerprints
- **Held-Out Test:** 1,111 fingerprints (official `validationData.csv` collected by unseen users/phones 4 months later)

In [2]:
data, prep = prepare_data(save=False)
print("Input WAPs:", len(prep["used_waps"]))
print("Train shape:", data["train"]["X"].shape)
print("Val shape:  ", data["val"]["X"].shape)
print("Test shape: ", data["test"]["X"].shape)

Input WAPs: 459
Train shape: (16432, 459)
Val shape:   (2795, 459)
Test shape:  (1111, 459)


## 2. Load the Master Hybrid Benchmark Results
The table below compares the four standardized experimental tracks on the held-out test partition ($N=1{,}111$):

In [3]:
results_path = METRICS_DIR / "hybrid_comparison.csv"
if results_path.exists():
    df_results = pd.read_csv(results_path)
    test_df = df_results[df_results["Split"] == "test"][[
        "Model", "building_acc", "floor_acc", "joint_acc",
        "mean_error_m", "median_error_m", "p90_error_m", "coord_mae_m", "evaal_score"
    ]].copy()
    test_df["building_acc"] = (test_df["building_acc"] * 100).round(2).astype(str) + "%"
    test_df["floor_acc"] = (test_df["floor_acc"] * 100).round(2).astype(str) + "%"
    test_df["joint_acc"] = (test_df["joint_acc"] * 100).round(2).astype(str) + "%"
    for col in ["mean_error_m", "median_error_m", "p90_error_m", "coord_mae_m", "evaal_score"]:
        test_df[col] = test_df[col].round(2)
    display(test_df)
else:
    print("Run train_hybrid.py first to generate metrics.")

                              Model building_acc  ...  coord_mae_m  evaal_score
1         Experiment A: DL Baseline       99.73%  ...         5.72         9.50
3  Experiment B: DL + Tree Stacking       99.82%  ...         5.65         9.31
5  Experiment C: DL + Residual GBDT       99.73%  ...         5.58         9.29
7    Full Hybrid: Stacked DL + GBDT       99.82%  ...         5.42         8.96

[4 rows x 9 columns]


## 3. Visualizing the Error Cumulative Distribution Functions (CDFs)
The CDF demonstrates that the Hybrid pipeline strictly dominates across the entire operational error range.

In [4]:
from IPython.display import Image
Image(filename=str(FIG_DIR / "hybrid_comparison_cdf.png"))

## 4. Error Distribution Boxplot (Percentile Comparison)

In [5]:
Image(filename=str(FIG_DIR / "hybrid_error_boxplot.png"))

## 5. End-to-End Inference Demo with Saved Hybrid System

In [6]:
system = HybridPositioningSystem.load(MODEL_DIR / "hybrid_system_best.joblib", device=device)
print("Hybrid System loaded in mode:", system.mode)

# Run inference on first 5 test samples
sample_x = data["test"]["X"][:5]
preds = system.predict(sample_x, prep, device=device)

demo_df = pd.DataFrame({
    "True Building": prep["building_encoder"].inverse_transform(data["test"]["building"][:5]),
    "Pred Building": preds["building"],
    "True Floor": prep["floor_encoder"].inverse_transform(data["test"]["floor"][:5]),
    "Pred Floor": preds["floor"],
    "True Longitude": data["test"]["coords_m"][:5, 0].round(2),
    "Pred Longitude": preds["coords_m"][:5, 0].round(2),
    "True Latitude": data["test"]["coords_m"][:5, 1].round(2),
    "Pred Latitude": preds["coords_m"][:5, 1].round(2),
    "Euclidean Error (m)": np.linalg.norm(data["test"]["coords_m"][:5] - preds["coords_m"], axis=1).round(2)
})
display(demo_df)

Hybrid System loaded in mode: full_hybrid
   True Building  Pred Building  ...  Pred Latitude  Euclidean Error (m)
0              1              1  ...     4864894.27                 9.54
1              2              2  ...     4864842.24                 4.13
2              2              2  ...     4864845.77                 5.91
3              2              2  ...     4864843.03                 0.59
4              0              0  ...     4864918.52                 3.88

[5 rows x 9 columns]
